# PrimeKG — Exploratory Data Analysis

EDA on the PrimeKG biomedical knowledge graph (genes, diseases, drugs, symptoms, and proteins), run against a Neo4j instance loaded with the dataset. Originally built as a Neo4j Browser dashboard; converted here into a runnable notebook.


In [ ]:
from neo4j import GraphDatabase
import pandas as pd
import os
from dotenv import load_dotenv

load_dotenv()

driver = GraphDatabase.driver(
    os.getenv("NEO4J_URI", "bolt://localhost:7687"),
    auth=(os.getenv("NEO4J_USERNAME", "neo4j"), os.getenv("NEO4J_PASSWORD"))
)

def run_query(query: str) -> pd.DataFrame:
    with driver.session(database=os.getenv("NEO4J_DATABASE", "neo4j")) as session:
        result = session.run(query)
        return pd.DataFrame([r.data() for r in result])


## Total nodes

*Chart type in original dashboard: table*


In [ ]:
query = """
MATCH (n) RETURN count(n) AS total_nodes;
"""
run_query(query)


## Total relations

*Chart type in original dashboard: table*


In [ ]:
query = """
MATCH ()-[r]->() RETURN count(r) AS total_relationships;
"""
run_query(query)


## Node types

*Chart type in original dashboard: pie*


In [ ]:
query = """
MATCH (n:Entity) RETURN n.node_type AS type, count(*) AS count ORDER BY count DESC;
"""
run_query(query)


## Relation types

*Chart type in original dashboard: bar*


In [ ]:
query = """
MATCH ()-[r]->() RETURN type(r) AS relation, count(*) AS count ORDER BY count DESC LIMIT 15;
"""
run_query(query)


## Node source breakdown

Shows which original databases (NCBI, DrugBank, MONDO, etc.) contributed the most nodes to PrimeKG.

*Chart type in original dashboard: pie*


In [ ]:
query = """
MATCH (n:Entity) RETURN n.node_source AS source, count(*) AS count ORDER BY count DESC;
"""
run_query(query)


## Cross-type relationship patterns

"Pathway patterns" in the graph — like gene→gene, drug→disease, gene→disease — showing the overall shape of how different entity types interact.

*Chart type in original dashboard: table*


In [ ]:
query = """
MATCH (a:Entity)-[r]->(b:Entity)
RETURN a.node_type AS from_type, type(r) AS relation, b.node_type AS to_type, count(*) AS count
ORDER BY count DESC
LIMIT 15;
"""
run_query(query)


## Most significant node

Node with the most overall connections.

*Chart type in original dashboard: table*


In [ ]:
query = """
MATCH (n:Entity)-[r]-()
RETURN n.node_name AS name, n.node_type AS type, count(r) AS connections
ORDER BY connections DESC
LIMIT 10;
"""
run_query(query)


## Most connected disease

Finds which single disease has the most connections overall — likely a well-studied, complex condition with lots of known genetic/drug/symptom links.

*Chart type in original dashboard: bar*


In [ ]:
query = """
MATCH (d:Entity {node_type: "disease"})-[r]-()
RETURN d.node_name AS disease, count(r) AS connections
ORDER BY connections DESC
LIMIT 10;
"""
run_query(query)


## Drug-disease direct connections

Shows specific examples of drugs directly linked to diseases, and what kind of relationship connects them (treats, causes, etc.).

*Chart type in original dashboard: table*


In [ ]:
query = """
MATCH (drug:Entity {node_type: "drug"})-[r]-(disease:Entity {node_type: "disease"})
RETURN drug.node_name AS drug, type(r) AS relation, disease.node_name AS disease
LIMIT 15;
"""
run_query(query)


## Gene-disease-drug triangle

Finds genes connected to both a disease and a drug — e.g. MT1A is linked to squamous cell carcinoma and hepatocellular carcinoma, and also connected to Copper — a genetic link suggesting Copper might have some biological relationship to these cancers, worth exploring further even though it's not an obvious/direct connection.

*Chart type in original dashboard: table*


In [ ]:
query = """
MATCH (gene:Entity {node_type: "gene/protein"})-[r1]-(disease:Entity {node_type: "disease"})
MATCH (gene)-[r2]-(drug:Entity {node_type: "drug"})
RETURN gene.node_name AS gene, disease.node_name AS disease, drug.node_name AS drug
LIMIT 15;
"""
run_query(query)


## Symptom-based disease similarity

Two different diseases that share a lot of the same symptoms — diseases with heavily overlapping symptoms can be genuinely hard for doctors to tell apart just by looking at what a patient is experiencing; this kind of analysis helps identify which diseases might get confused with each other, or which ones might share an underlying biological cause.

*Chart type in original dashboard: table*


In [ ]:
query = """
MATCH (d1:Entity {node_type: "disease"})-[:disease_phenotype_positive]->(p:Entity)<-[:disease_phenotype_positive]-(d2:Entity {node_type: "disease"})
WHERE d1.node_name < d2.node_name
WITH d1, d2, count(p) AS shared_symptoms
WHERE shared_symptoms > 10
RETURN d1.node_name AS disease_1, d2.node_name AS disease_2, shared_symptoms
ORDER BY shared_symptoms DESC
LIMIT 10;
"""
run_query(query)


## Drug safety check

For real drugs, shows their approved uses, off-label uses, AND conditions where they should be avoided — all three sides of a drug's clinical profile in one view.

*Chart type in original dashboard: table*


In [ ]:
query = """
MATCH (drug:Entity {node_type: "drug"})-[r]-(d:Entity {node_type: "disease"})
WHERE type(r) IN ["indication", "off_label_use", "contraindication"]
RETURN drug.node_name AS drug, type(r) AS relation, d.node_name AS disease
LIMIT 200;
"""
run_query(query)


## Drug-drug interaction relation types

Checks if there's a specific drug-drug interaction relationship type — drug interactions (which combinations are dangerous together) is a genuinely important, distinct medical question.

*Chart type in original dashboard: pie*


In [ ]:
query = """
MATCH ()-[r]->() WHERE type(r) CONTAINS "drug" RETURN DISTINCT type(r) AS relation_type, count(*) AS count ORDER BY count DESC;
"""
run_query(query)


## Most significant node for drug-drug interactions

Which drugs interact dangerously when taken together.

*Chart type in original dashboard: table*


In [ ]:
query = """
MATCH (d:Entity {node_type: "drug"})-[r:drug_drug]-()
RETURN d.node_name AS drug, count(r) AS interaction_count
ORDER BY interaction_count DESC
LIMIT 10;
"""
run_query(query)


## Graph Visualizations

The following queries return subgraphs best viewed as a graph visualization (e.g. in Neo4j Browser) rather than a table. They're included here for completeness — running them in this notebook returns raw node/relationship data rather than a rendered graph.


### Specific drug interactions (example: Quinidine)


In [ ]:
query = """
MATCH (d:Entity {node_name: "Quinidine", node_type: "drug"})-[r:drug_drug]-(other:Entity)
RETURN d, r, other
LIMIT 30;
"""
run_query(query)


### Visualization of most significant node (example: multi-cellular organism)


In [ ]:
query = """
MATCH (n:Entity {node_name: "multi-cellular organism"})-[r]-(m:Entity)
RETURN n, r, m
LIMIT 30;
"""
run_query(query)


### Overall cross-type relationship view


In [ ]:
query = """
MATCH (n:Entity)-[r]->(m:Entity)
WHERE n.node_type <> m.node_type
RETURN n, r, m
LIMIT 150;
"""
run_query(query)


### Specific disease comparison (shared phenotypes)


In [ ]:
query = """
MATCH (d1:Entity {node_name: "X-linked intellectual disability"})-[r1:disease_phenotype_positive]->(p:Entity)<-[r2:disease_phenotype_positive]-(d2:Entity {node_name: "developmental and epileptic encephalopathy"})
RETURN d1, r1, p, r2, d2
LIMIT 10;
"""
run_query(query)
